In [ ]:
from dataclasses import asdict

import cv2
import numpy as np

from uavcalibration.datasets import VisLocDataset
from uavcalibration.calibration import Calibration
from uavcalibration.rectification import *
from uavcalibration.matching import *

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")

In [ ]:
# match_images(image0, image1)
uav_data = dataset[0]

calibration = Calibration(uav_data.uav_image)
calibration.coarse_calibrate(**asdict(uav_data))
src_shape = uav_data.uav_image.shape
calibration.transform.adjust_shape(src_shape=(src_shape[1],src_shape[0]))
image0 = calibration.calibrated_image
image1 = uav_data.satellite_image

print(image0.shape, image1.shape)

In [ ]:
match_result = match_images(
    image0,
    image1,
    method=MatchingMethod.LIGHTGLUE,
    max_kpts0=2048,
    max_kpts1=2048,
)
plot_matches(match_result, image0, image1)

In [ ]:
resolution = uav_data.height / uav_data.focal_length  # meters per pixel
tolerance = 5  # meters
threshold = tolerance / resolution  # tolerance threshold
homography_result = match_homography(match_result.kpts0, match_result.kpts1, threshold)

plot_matches(homography_result, image0, image1)